# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the [FAIR^2 dataset: Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the [`mlcroissant`](https://github.com/mlcommons/croissant-python) library.

## Dataset Source

- Schema: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)
- Description: This dataset details the analysis of protein abundance, modifications, and sequences in human (Homo sapiens) samples. Includes accession, description, coverage, peptide counts, molecular weight, and normalized abundances, with annotation fields across multiple samples.


In [ ]:
# Ensure mlcroissant is available (restart the kernel if newly installed)
!pip install mlcroissant --quiet

## 1. Data Loading

Load the Croissant dataset metadata and inspect the main dataset information.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display dataset level metadata
md = dataset.metadata
print(f"DATASET NAME: {md.name}")
print(f"DESCRIPTION: {md.description}")
print(f"VERSION: {getattr(md, 'version', 'N/A')}")
print(f"Published: {getattr(md, 'datePublished', 'N/A')}")

## 2. Data Overview

Review available record sets and their associated field and column `@id`s.

Each **Record Set** corresponds to a logical table exposed by the dataset.

In [ ]:
# Collect and show all record sets by their @id and name
record_sets = list(dataset.metadata.record_sets)
if not record_sets:
    print('No explicit record sets found. The dataset may have a single implicit record set.')
else:
    print("Available record sets:")
    for rec in record_sets:
        print(f"- @id: {rec['@id']}, name: {rec.get('name', 'N/A')}")

In [ ]:
# If no explicit record sets, try to preview records for main dataset
# Otherwise, preview first record from the first record set by @id
if not record_sets:
    example_record_set_id = None
    try:
        records = list(dataset.records())
        print(f"Sample record with default records(): {records[0]}")
    except Exception as e:
        print(f"Dataset cannot be iterated: {e}")
else:
    example_record_set_id = record_sets[0]['@id']
    print(f"Sample rows for record set @id = {example_record_set_id}:")
    for i, x in enumerate(dataset.records(record_set=example_record_set_id)):
        print(json.dumps(x, indent=2))
        if i > 1:
            break

### Field and Column IDs

Show available field `@id`s for the main record set for precise referencing in later steps.

In [ ]:
def pretty_field_info(record_set):
    print('Fields in record set:')
    for field in record_set.get('fields', []):
        print(f"  - @id: {field['@id']}, name: {field.get('name', 'N/A')}, dataType: {field.get('dataType', 'N/A')}")
    print('Columns in record set:')
    for column in record_set.get('columns', []):
        print(f"  - @id: {column['@id']}, name: {column.get('name', 'N/A')}, dataType: {column.get('dataType', 'N/A')}")

if record_sets and example_record_set_id:
    record_set_map = {rec['@id']: rec for rec in record_sets}
    rs = record_set_map[example_record_set_id]
    pretty_field_info(rs)

## 3. Data Extraction

Extract all records from each record set as DataFrames using their `@id`s for programmatic access and manipulation.

In [ ]:
# List all record sets by their @id (from previous cells)
if record_sets and len(record_sets) > 0:
    record_set_ids = [r['@id'] for r in record_sets]
else:
    # Try single implicit record set (use None for default)
    record_set_ids = [None]

dataframes = {}
for rs_id in record_set_ids:
    try:
        if rs_id is not None:
            recs = list(dataset.records(record_set=rs_id))
            df = pd.DataFrame(recs)
        else:
            recs = list(dataset.records())
            df = pd.DataFrame(recs)
        dataframes[rs_id if rs_id is not None else 'main'] = df
        print(f"Loaded DataFrame for record set @id = {rs_id if rs_id else 'main'} with shape {df.shape}")
        print(f"Fields: {df.columns.tolist()}")
        display(df.head())
    except Exception as e:
        print(f"Error loading record set {rs_id}: {e}")

# For subsequent steps, select the first record set actually loaded
record_set_in_use = next(iter(dataframes.keys()))

## 4. Exploratory Data Analysis (EDA)

Perform typical filtering, normalization, and grouping operations using only fields referenced by their `@id`.

In [ ]:
df = dataframes[record_set_in_use]
print(f"Using record set: {record_set_in_use}")

# Example: Find numeric fields from first few records (Croissant always references fields by @id)
print("Numeric-like fields from first row:")
for col in df.columns:
    # Try to infer if the column is numeric
    vals_sample = df[col].dropna().head(10)
    try:
        vals_numeric = pd.to_numeric(vals_sample)
        if vals_numeric.notna().all():
            print(f"  - {col}")
    except Exception:
        pass

# Pick a numeric field's @id for demonstration (e.g., coverage, MW, or similar)
example_numeric_field = None
for col in df.columns:
    vals_sample = df[col].dropna().head(10)
    try:
        vals_numeric = pd.to_numeric(vals_sample)
        if vals_numeric.notna().all():
            example_numeric_field = col
            break
    except Exception:
        continue

if example_numeric_field:
    # Use this field for filtering/normalizing
    numeric_field = example_numeric_field
    print(f"Analyzing numeric field with @id: {numeric_field}")
    # Convert to numeric if not already
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].median()  # Example: use median as threshold
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered to {len(filtered_df)} records with {numeric_field} > {threshold:.2f}")
    display(filtered_df[[numeric_field]].head())

    # Normalizing
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print("After normalization, first rows:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a candidate categorical field (choose string/object type)
    candidate_groups = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
    group_field = candidate_groups[0] if candidate_groups else None
    if group_field and group_field in df.columns:
        print(f"Grouping by {group_field} (using @id)")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        display(grouped_df.head())
    else:
        print("No obvious categorical group field found.")
else:
    print("No numeric field found for EDA.")

## 5. Visualization

Visualize the distribution of a numeric field and group-wise statistics if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'numeric_field' in locals() and numeric_field in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No visualization generated (no numeric field found).")

## 6. Conclusion

This notebook demonstrated loading and exploring the FAIR^2 Croissant dataset using `mlcroissant`, performing field-level access by `@id` as recommended by the standard.

- We programmatically loaded dataset record sets and referenced all data columns and attributes by their `@id`.
- We illustrated EDA practices such as detecting numeric and categorical fields, filtering, normalization, and basic visualizations.

**Next steps:** Apply domain-specific transformations or downstream analytics, always referencing Croissant entities (fields, columns, record sets) by their identities for full reproducibility and interoperability.